<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/Exercises_MCP_LLM_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises (Student) - MCP Client with LLM

In [ ]:
!pip install -q mcp nest_asyncio requests

In [41]:
import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set

In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [34]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@mcp.tool()
def greet(name: str) -> str:
    """Return a greeting string."""
    return f"Hello, {name}!"

@mcp.resource("greeting://welcome")
def get_greeting() -> str:
    return "Welcome to the MCP Exercise!"

if __name__ == "__main__":
    # On force explicitement le transport stdio pour éviter les conflits dans Colab
    mcp.run(transport="stdio")

Overwriting server.py


## Exercise 1 (provide answer)

### Pourquoi le transport STDIO est-il plus simple ?

Le transport **STDIO** (Standard Input/Output) est idéal pour le développement local pour plusieurs raisons :
- **Absence de serveur réseau** : Contrairement au HTTP, il n'y a pas besoin de gérer des adresses IP, des ports (comme `localhost:8000`) ou des certificats SSL.
- **Communication Directe** : Le client lance le serveur comme un sous-processus. Ils communiquent directement via les flux d'entrée/sortie standards.
- **Zéro Configuration** : Aucune règle de pare-feu à modifier ou de ports à ouvrir.
- **Débogage facilité** : Si le processus serveur s'arrête, la connexion se ferme immédiatement, ce qui rend le cycle de développement plus rapide.

## Exercise 2

In [12]:
import asyncio
import nest_asyncio
try:
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client
except ImportError:
    !pip install -q mcp
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    params = StdioServerParameters(command="mcp", args=["run", "server.py"])
    # On désactive errlog pour Colab
    async with stdio_client(params, errlog=False) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            print("Connection status: Initialized")

In [ ]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


Exercise 2: OK (connected and initialized)


## Exercise 3

In [13]:
async def ex3_list():
    params = StdioServerParameters(command="mcp", args=["run", "server.py"])
    async with stdio_client(params, errlog=False) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

            resources_resp = await session.list_resources()
            print("--- RESOURCES ---")
            for res in resources_resp.resources:
                print(f"- {res.name}: {res.uri}")

            tools_resp = await session.list_tools()
            print("\n--- TOOLS ---")
            for t in tools_resp.tools:
                print(f"Tool: {t.name}")
                print(f"Description: {t.description}")
                print(f"Properties: {t.inputSchema.get('properties', {})}")
                print("-" * 20)

In [ ]:
await ex3_list()

RESOURCES: meta=None nextCursor=None resources=[]
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Exercise 4

### Explication : Conversion vers LLM Tool

La conversion repose sur le fait que MCP utilise **JSON Schema** pour définir les arguments des outils dans `inputSchema`. Les LLM (comme GPT-4) attendent exactement cette structure pour leur mécanisme de **Function Calling**.

La fonction `convert_to_llm_tool` extrait :
1. Le **nom** de l'outil.
2. La **description** (utilisée par le LLM pour savoir quand appeler l'outil).
3. Le **schéma des paramètres**, permettant au LLM de générer des arguments typés (int, string, etc.) au format JSON.

In [4]:
def convert_to_llm_tool(tool):
    """Convertit un outil MCP en format standard OpenAI Function Calling."""
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "MCP tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }

## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [39]:
import asyncio
import json
import nest_asyncio
import os
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls

In [36]:
from typing import List, Dict, Any
import mcp.types as types
import traceback
import asyncio
import sys
import os

def stub_plan(prompt: str, functions: List[Dict[str, Any]]):
    if "add" in prompt.lower():
        return [{"name": "add", "args": {"a": 2, "b": 20}}]
    if "multiply" in prompt.lower():
        return [{"name": "multiply", "args": {"a": 3, "b": 4}}]
    return []

def print_error_recursive(e, depth=0):
    indent = "  " * depth
    if hasattr(e, "exceptions"):
        print(f"{indent}[Group]: {type(e).__name__}")
        for sub in e.exceptions:
            print_error_recursive(sub, depth + 1)
    else:
        print(f"{indent}Actual Root Cause -> {type(e).__name__}: {e}")

async def ex5_run(prompt: str = "Add 2 to 20"):
    server_path = os.path.abspath("server.py")
    params = StdioServerParameters(command=sys.executable, args=[server_path])

    try:
        async with stdio_client(params, errlog=False) as (r, w):
            async with ClientSession(r, w) as session:
                await asyncio.sleep(2)
                await session.initialize()

                tools_resp = await session.list_tools()
                functions = [convert_to_llm_tool(t) for t in tools_resp.tools]

                calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
                print(f"\nPrompt: {prompt}")

                for call in calls:
                    print(f"Executing: {call['name']} with {call['args']}")
                    result = await session.call_tool(call['name'], arguments=call['args'])

                    texts = []
                    for content in result.content:
                        if hasattr(content, 'text'):
                            texts.append(content.text)
                        else:
                            texts.append(str(content))

                    print(f"Result from MCP: {' '.join(texts)}")
    except BaseException as e:
        print(f"\n--- Detailed Error Unpacking ---")
        print_error_recursive(e)

In [ ]:
await ex5_run("Add 2 to 20")

tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result: ['22']


## Optional - add multiply(a, b) and rerun

In [42]:
print("--- TEST FINAL (EXECUTION) ---")
try:
    # Ensure global variable is available
    if 'USE_REAL_LLM' not in globals():
        USE_REAL_LLM = False

    print("\nTest 1: Multiplication")
    await ex5_run("Multiply 3 and 4")
    print("\n" + "="*30)
    print("\nTest 2: Addition")
    await ex5_run("Add 2 to 20")
except Exception as e:
    print(f"\nErreur critique lors de l'appel : {e}")

--- TEST FINAL (EXECUTION) ---

Test 1: Multiplication

Prompt: Multiply 3 and 4
Executing: multiply with {'a': 3, 'b': 4}
Result from MCP: 12


Test 2: Addition

Prompt: Add 2 to 20
Executing: add with {'a': 2, 'b': 20}
Result from MCP: 22
